# 2. Structures

A structure places copies of its components in a shape: a ball, a rod, a ring, a vesicle, or a lattice of other structures.

For each structure in this notebook we:

1. set it up
2. call `.assemble()`, which writes its `.gro` file (no GROMACS needed)
3. call `.export_to()`, which copies the files to a folder you choose
4. look at the result

At the end you can download all exports as one zip, to open them in VMD or PyMOL.

Tip: `help(Ball)`, `help(Vesicle)` and so on list all settings of a structure.

In [ ]:
# @title Setup: on Colab, install Shapespyer (about 1 min)
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
CONFIG_FILE = Path("shapespyer.yaml")  # where Shapespyer keeps its files

if IN_COLAB:
    %pip install -q "shapespyer==0.3.0"

    # nglview, behind .show(), draws molecules through Colab's widget manager
    from google.colab import output
    output.enable_custom_widget_manager()

    # the library, Temp and Projects go to /content, to browse them in the Files panel
    from shapes.project import Config
    config = Config(CONFIG_FILE)
    config.app_path = "/content/shapespyer"
    config.save()
elif not CONFIG_FILE.is_file():
    err_msg = f"No {CONFIG_FILE} in {Path.cwd()}; run the notebook in its own folder, next to shapespyer.yaml"
    raise FileNotFoundError(err_msg)

from shapes.project.files import AppFileManager

# the first call sets where the files go, for the whole session
print("Shapespyer files:", AppFileManager(CONFIG_FILE).config.app_path)

In [ ]:
import shutil
from pathlib import Path

import nglview

from shapes.basics.utils import LogConfiguration
from shapes.project import Ball, Lattice, Library, Ring, Rod, Surfactant, Vesicle
from shapes.project.files import AppFileManager

LogConfiguration("WARNING")

library = Library()
library.clean_temp()
EXPORTS = Path("structure_exports").resolve()


def export_and_show(structure, name):
    """Export a structure to structure_exports/<name>/ and draw its .gro."""
    directory = EXPORTS / name
    structure.export_to(directory, overwrite=True)
    gro = next(directory.glob("*.gro"))
    print(gro)
    return nglview.show_structure_file(str(gro))

### Ball

An SDS micelle. The same with the command-line tool:
`shape --dio=inp,usr-out -i config_SO4C12_smiles.gro -o configS -x .gro -s ball -r SDS --mint=41 -l 16 --dmin=0.4`

- `rmin`: radius of the empty space in the centre, in nm
- `dmin`: smallest distance between the "bone" atoms of neighbour molecules, in nm
- `mint`: index of the atom that points to the centre

Try `ball.fill = "fibo"` for a Fibonacci-sphere packing.

In [ ]:
ball = Ball()
ball.components.append(Surfactant.load("SDS"))
ball.layers[0].rmin = 1.3
ball.layers[0].dmin = 0.4
ball.components["SDS"].mint = 41

ball.assemble()
export_and_show(ball, "ball")

### Rod

The same with the command-line tool:
`shape --dio=usr-out -i config_SO4C12_smiles.gro -o configS -x .gro -s rod -r SDS -l 15 -t 10`

In [ ]:
rod = Rod()
rod.components.append(Surfactant.load("SDS"))
rod.rmin = 1.23
rod.dmin = 0.4
rod.turns = 12

rod.assemble()
export_and_show(rod, "rod")

### Ring

The same with the command-line tool:
`shape --dio=inp,usr-out -i SDS-single1.gro -o config1 -x .gro -s ring -r SDS -l 15`

Try `ring.fxz = True` (flatten each molecule in the XZ plane first, so it spreads less along Y) or `ring.rev = True` (swap the inner and outer atoms of each molecule).

Remember: when you change a setting, call `.assemble()` again to write a new `.gro`.

In [ ]:
ring = Ring()
ring.components.append(Surfactant.load("SDS"))
ring.rmin = 1.2

ring.assemble()
export_and_show(ring, "ring")

### Vesicle

The same with the command-line tool:
`shape --dio=inp,usr-out -i config_SO4C12_smiles.gro -o configS -x .gro -s ves -l 16 -r SDS --mint=41 --rev`

A vesicle is made of lamellae, each with an inner and an outer leaflet. Notebook 3 builds an asymmetric one.

In [ ]:
vesicle = Vesicle()
vesicle.components.append(Surfactant.load("SDS"))
vesicle.lamellae[0].rmin = 1.23
vesicle.lamellae[0].inner_leaflet_dmin = 0.4
vesicle.fill = "fibo"
vesicle.components["SDS"].mint = 41

vesicle.assemble()
export_and_show(vesicle, "vesicle")

### Lattice

A lattice repeats a unit structure on its nodes, here an SDS ball on a face-centred cubic lattice. Other types are `"LCUB"` (simple cubic) and `"LBCC"` (body-centred cubic).

The lattice builds its unit itself, so the unit isn't assembled here.

In [ ]:
unit = Ball()
unit.components.append(Surfactant.load("SDS"))
unit.layers[0].rmin = 1.3
unit.layers[0].dmin = 0.4

lattice = Lattice(unit=unit)
lattice.nlatx = 2
lattice.nlaty = 1
lattice.nlatz = 1
lattice.lattice_type = "LFCC"

lattice.assemble()
export_and_show(lattice, "lattice")

### Download the exports

On Colab, this zips `structure_exports/` and downloads it. Locally, the zip is next to the notebook.

In [ ]:
archive = shutil.make_archive("structure_exports", "zip", EXPORTS)
if IN_COLAB:
    from google.colab import files
    files.download(archive)
archive

### Clean up (optional)

This deletes the exports, the zip and Temp, where the structures were assembled. `force=True` is needed for Temp, because the structures of this session still hold their files there.

In [ ]:
AppFileManager().delete_dir(EXPORTS, force=True)
Path(archive).unlink(missing_ok=True)
library.clean_temp(force=True)
print(EXPORTS.exists())